In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

In [ ]:
train_dir = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"
val_dir   = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing"

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),

    # 🔥 ALL augmentations BEFORE ToTensor
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)),

    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=val_transform)

print("Class Mapping:", train_dataset.class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = models.efficientnet_b0(pretrained=True)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
import torch

weights = torch.tensor([2.1, 1.0, 1.0, 1.0]).to(device)  # glioma higher
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # 🔷 TRAINING
        model.train()
        train_loss = 0
        train_correct = 0
        total = 0

        train_bar = tqdm(train_loader, desc="Training", leave=False)

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # metrics
            _, preds = torch.max(outputs, 1)
            correct = (preds == labels).sum().item()

            train_correct += correct
            total += labels.size(0)
            train_loss += loss.item()

            # update tqdm bar
            train_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{(train_correct/total):.4f}"
            })

        train_acc = train_correct / total
        train_loss /= len(train_loader)

        # 🔷 VALIDATION
        model.eval()
        val_correct = 0
        total = 0

        val_bar = tqdm(val_loader, desc="Validation", leave=False)

        with torch.no_grad():
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                _, preds = torch.max(outputs, 1)

                correct = (preds == labels).sum().item()

                val_correct += correct
                total += labels.size(0)

                val_bar.set_postfix({
                    "acc": f"{(val_correct/total):.4f}"
                })

        val_acc = val_correct / total

        # 🔷 Epoch Summary
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Acc: {val_acc:.4f}")

In [ ]:
train_model(model, train_loader, val_loader, epochs=10)

In [ ]:
# Unfreeze model
for param in model.parameters():
    param.requires_grad = True

# FIXED optimizer
optimizer = optim.Adam([
    {"params": model.features.parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4}
])

train_model(model, train_loader, val_loader, epochs=10)

In [ ]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

In [ ]:
torch.save(model.state_dict(), "final_model_.pth")